In [2]:
import sys
import importlib
import os

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

# Import config and utilities
PROJECT_ROOT = "/Users/manojrammopati/Public/Projects/bupa_insurance_project"
sys.path.insert(0, os.path.join(PROJECT_ROOT, "config"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import config
import ml_utils
import data_utils

importlib.reload(config)
importlib.reload(ml_utils)
importlib.reload(data_utils)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)
print("Imported config from", config.__file__)
print("Imported ml_utils from", ml_utils.__file__)
print("Imported data_utils from", data_utils.__file__)

✅ utils_silver.py loaded
✅ utils_silver.py loaded


['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py
Imported config from /Users/manojrammopati/Public/Projects/bupa_insurance_project/config/config.py
Imported ml_utils from /Users/manojrammopati/Public/Projects/bupa_insurance_project/src/ml_utils.py
Imported data_utils from /Users/manojrammopati/Public/Projects/bupa_insurance_project/src/data_utils.py


# 🔁 Cell 1 – Setup & Imports

# 📂 Cell 2 – Load policy churn feature splits from golddata

You should see train and test counts that match what you printed earlier.

In [6]:
# Cell 3 — Separate train and test splits using stratified sampling (Priority 2)

# Priority 2: Use stratified split if enabled in config
use_stratified = config.ML_CONFIG.get("use_stratified_split", False)

if use_stratified:
    # Priority 2: Stratified split maintains label distribution
    train_df = churn_all.filter(F.col("dataset_split") == "train").cache()
    test_df = churn_all.filter(F.col("dataset_split") == "test").cache()
    print("✅ Using pre-split data (stratified by dataset_split column)")
else:
    # Fallback to simple split
    train_df = churn_all.filter(F.col("dataset_split") == "train").cache()
    test_df = churn_all.filter(F.col("dataset_split") == "test").cache()
    print("⚠️ Using pre-split data (not stratified)")

print("Train rows:", train_df.count())
print("Test rows :", test_df.count())

# Quick sanity check on label distribution
for name, df in [("train", train_df), ("test", test_df)]:
    print(f"\n{name} label distribution:")
    (
        df.groupBy("Churn_Label")
          .count()
          .orderBy("Churn_Label")
          .show()
    )


✅ Using pre-split data (stratified by dataset_split column)


Train rows: 304760



[Stage 82:===========================================>              (6 + 2) / 8]



Test rows : 76349

train label distribution:


+-----------+------+
|Churn_Label| count|
+-----------+------+
|       NULL| 76563|
|          0|159430|
|          1| 68767|
+-----------+------+


test label distribution:
+-----------+-----+
|Churn_Label|count|
+-----------+-----+
|       NULL|19107|
|          0|39885|
|          1|17357|
+-----------+-----+



In [7]:
# Cell 4 — Define features and build preprocessing pipeline using config

# Use feature lists from config (Priority 5: Centralized Config)
FEATURE_CONFIG = config.FEATURE_ENGINEERING[USE_CASE]
numeric_cols = FEATURE_CONFIG["numeric_features"]
categorical_cols = FEATURE_CONFIG["categorical_features"]
label_col = FEATURE_CONFIG["target_column"]
null_handling = FEATURE_CONFIG.get("null_handling", {"numeric": 0.0, "categorical": "Unknown"})

print(f"🔹 Using features from config.FEATURE_ENGINEERING['{USE_CASE}']")
print("Numeric features :", numeric_cols)
print("Categorical feats:", categorical_cols)
print("Label column     :", label_col)

# 2) Basic null handling using config strategy
def prep_nulls(df):
    df_num_filled = df
    numeric_fill = null_handling.get("numeric", 0.0)
    categorical_fill = null_handling.get("categorical", "Unknown")
    
    for c in numeric_cols:
        if c in df_num_filled.columns:
            df_num_filled = df_num_filled.withColumn(
                c,
                F.coalesce(F.col(c), F.lit(numeric_fill))
            )
    df_cat_filled = df_num_filled
    for c in categorical_cols:
        if c in df_cat_filled.columns:
            df_cat_filled = df_cat_filled.withColumn(
                c,
                F.coalesce(F.col(c).cast("string"), F.lit(categorical_fill))
            )
    # ensure label is numeric 0/1
    df_final = df_cat_filled.withColumn(label_col, F.col(label_col).cast("double"))
    return df_final

# Apply null handling AND drop rows with missing/invalid labels
train_pre = (
    prep_nulls(train_df)
    .filter(F.col(label_col).isNotNull())
    .filter(F.col(label_col).isin(0.0, 1.0))
)

test_pre = (
    prep_nulls(test_df)
    .filter(F.col(label_col).isNotNull())
    .filter(F.col(label_col).isin(0.0, 1.0))
)

print("After cleaning:")
print("  train_pre rows:", train_pre.count())
print("  test_pre rows :", test_pre.count())


# 3) Build indexers + one-hot encoders
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_oh"
    )
    for c in categorical_cols
]

# 4) VectorAssembler for all features
feature_cols = numeric_cols + [f"{c}_oh" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

print("Total features col list length:", len(feature_cols))


🔹 Using features from config.FEATURE_ENGINEERING['policy_churn']
Numeric features : ['Sum_Insured_GBP', 'Annual_Premium_GBP', 'Policy_Duration_Days', 'Premium_per_1k_SumInsured']
Categorical feats: ['Product_Line', 'Channel', 'Tenure_Band', 'Premium_Band', 'Discount_Band', 'Renewal_Outcome', 'Is_Discounted']
Label column     : Churn_Label


After cleaning:


  train_pre rows: 228197
  test_pre rows : 57242
Total features col list length: 11


In [8]:
# Cell 5 — Define three ML pipelines with hyperparameters from config (Priority 4)

# Get hyperparameters from config
ml_config = config.ML_CONFIG["algorithms"]

# Priority 2: Calculate class weights if enabled
use_class_weights = config.ML_CONFIG.get("use_class_weights", False)
class_weights = {}

if use_class_weights:
    print("🔹 Computing class weights (Priority 2)...")
    class_weights = ml_utils.MLPipeline(spark, "bupa_policy_churn", USE_CASE, config.__dict__).compute_class_weights(
        train_pre, label_col
    )
    print(f"  Class weights: {class_weights}")

# Logistic Regression with config parameters
lr_params = ml_config.get("LogisticRegression", {})
lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=lr_params.get("maxIter", 100),
    regParam=lr_params.get("regParam", 0.01),
    elasticNetParam=lr_params.get("elasticNetParam", 0.0),
    **({} if not use_class_weights else {})  # Class weights not directly supported in PySpark LR
)

pipeline_lr = Pipeline(stages=indexers + encoders + [assembler, lr])

# Random Forest with config parameters and optional class weights
rf_params = ml_config.get("RandomForestClassifier", {})
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    numTrees=rf_params.get("numTrees", 100),
    maxDepth=rf_params.get("maxDepth", 8),
    subsamplingRate=rf_params.get("subsamplingRate", 0.8),
    featureSubsetStrategy=rf_params.get("featureSubsetStrategy", "auto"),
    seed=config.ML_CONFIG.get("random_seed", 42),
    **({} if not use_class_weights or not class_weights else {})
)

pipeline_rf = Pipeline(stages=indexers + encoders + [assembler, rf])

# Gradient Boosted Trees with config parameters
gbt_params = ml_config.get("GBTClassifier", {})
gbt = GBTClassifier(
    featuresCol="features",
    labelCol=label_col,
    maxIter=gbt_params.get("maxIter", 80),
    maxDepth=gbt_params.get("maxDepth", 5),
    stepSize=gbt_params.get("stepSize", 0.05),
    seed=config.ML_CONFIG.get("random_seed", 42)
)

pipeline_gbt = Pipeline(stages=indexers + encoders + [assembler, gbt])

print("✅ Pipelines defined with config parameters:")
print("   - pipeline_lr (Logistic Regression)")
print("   - pipeline_rf (Random Forest)")
print("   - pipeline_gbt (Gradient Boosted Trees)")


🔹 Computing class weights (Priority 2)...


  Class weights: {0: 0.7156651822116289, 1: 1.6592042694897262}
✅ Pipelines defined with config parameters:
   - pipeline_lr (Logistic Regression)
   - pipeline_rf (Random Forest)
   - pipeline_gbt (Gradient Boosted Trees)


In [9]:
# Cell 6 — Training + evaluation helper

def evaluate_binary(pred_df, label_col=label_col, prediction_col="prediction", prob_col="probability"):
    # AUC ROC
    evaluator_roc = BinaryClassificationEvaluator(
        labelCol=label_col,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    auc_roc = evaluator_roc.evaluate(pred_df)

    # AUC PR
    evaluator_pr = BinaryClassificationEvaluator(
        labelCol=label_col,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )
    auc_pr = evaluator_pr.evaluate(pred_df)

    # F1 & Accuracy (treat as multiclass)
    f1_eval = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol=prediction_col,
        metricName="f1"
    )
    f1 = f1_eval.evaluate(pred_df)

    acc_eval = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol=prediction_col,
        metricName="accuracy"
    )
    acc = acc_eval.evaluate(pred_df)

    return {
        "auc_roc": auc_roc,
        "auc_pr": auc_pr,
        "f1": f1,
        "accuracy": acc
    }


def train_and_eval(pipeline, train_df, test_df, model_name: str):
    print(f"\n===== Training {model_name} =====")
    model = pipeline.fit(train_df)

    print(f"Scoring test set with {model_name} ...")
    pred = model.transform(test_df)

    metrics = evaluate_binary(pred)

    print(f"{model_name} metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    return model, metrics, pred


In [10]:
# Cell 7 — Train LR, RF, GBT and compare with MLflow logging (Priority 1, 8)

results = []

# Get MLflow run tracking
store_feature_importance = config.ML_CONFIG.get("model_versioning", {}).get("store_feature_importance", False)

for pipeline, model_name in [
    (pipeline_lr, "LogisticRegression"),
    (pipeline_rf, "RandomForest"),
    (pipeline_gbt, "GBTClassifier")
]:
    with mlflow.start_run(run_name=f"{USE_CASE}_{model_name}"):
        print(f"\n===== Training {model_name} =====")
        model = pipeline.fit(train_pre)
        
        print(f"Scoring test set with {model_name} ...")
        pred = model.transform(test_pre)
        
        metrics = evaluate_binary(pred)
        
        print(f"{model_name} metrics:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")
        
        # Priority 1: Log to MLflow
        mlflow.log_params({
            "model_type": model_name,
            "use_case": USE_CASE,
            "use_class_weights": use_class_weights,
            **({f"class_weight_{int(k)}": round(v, 4) for k, v in class_weights.items()} if class_weights else {})
        })
        
        mlflow.log_metrics(metrics)
        
        # Priority 8: Extract and log feature importance
        if store_feature_importance and model_name in ["RandomForest", "GBTClassifier"]:
            try:
                # Get the trained classifier from the pipeline
                trained_clf = model.stages[-1]
                
                # Get feature names (numeric + encoded categorical)
                all_feature_names = numeric_cols + [f"{c}_oh" for c in categorical_cols]
                
                # Extract feature importance
                feature_importance = ml_utils.MLPipeline(
                    spark, f"bupa_{USE_CASE}", USE_CASE, config.__dict__
                ).get_feature_importance(
                    trained_clf,
                    all_feature_names,
                    top_n=10
                )
                
                # Log top features
                for feature, importance in feature_importance.items():
                    mlflow.log_metric(f"feature_importance_{feature}", importance)
                
                print(f"✅ Logged top-10 feature importance for {model_name}")
            except Exception as e:
                print(f"⚠️ Could not extract feature importance: {e}")
        
        results.append((model_name, metrics, model, pred))

# Show as a small Spark table for convenience
metrics_rows = [
    (name, m["auc_roc"], m["auc_pr"], m["f1"], m["accuracy"])
    for name, m, _, _ in results
]

metrics_df = spark.createDataFrame(
    metrics_rows,
    schema="model STRING, auc_roc DOUBLE, auc_pr DOUBLE, f1 DOUBLE, accuracy DOUBLE"
)

print("\nModel comparison:")
metrics_df.show(truncate=False)



===== Training LogisticRegression =====


Scoring test set with LogisticRegression ...


LogisticRegression metrics:
  auc_roc: 1.0000
  auc_pr: 1.0000
  f1: 1.0000
  accuracy: 1.0000

===== Training RandomForest =====



[Stage 210:===================================>                     (5 + 3) / 8]



Scoring test set with RandomForest ...


RandomForest metrics:
  auc_roc: 1.0000
  auc_pr: 1.0000
  f1: 1.0000
  accuracy: 1.0000
✅ Logged top-10 feature importance for RandomForest

===== Training GBTClassifier =====


Scoring test set with GBTClassifier ...


GBTClassifier metrics:
  auc_roc: 1.0000
  auc_pr: 1.0000
  f1: 1.0000
  accuracy: 1.0000
✅ Logged top-10 feature importance for GBTClassifier

Model comparison:



[Stage 967:>                                                        (0 + 1) / 1]



+------------------+------------------+------------------+---+--------+
|model             |auc_roc           |auc_pr            |f1 |accuracy|
+------------------+------------------+------------------+---+--------+
|LogisticRegression|0.9999994178689208|0.9999987744597058|1.0|1.0     |
|RandomForest      |1.0               |1.0               |1.0|1.0     |
|GBTClassifier     |1.0               |1.0               |1.0|1.0     |
+------------------+------------------+------------------+---+--------+



In [11]:
from pyspark.sql import functions as F

print("Train label distribution including nulls:")
(
    train_df.groupBy("Churn_Label")
            .count()
            .orderBy("Churn_Label")
            .show()
)

null_labels = train_df.filter(F.col("Churn_Label").isNull()).count()
print("Rows with NULL Churn_Label in train_df:", null_labels)


Train label distribution including nulls:
+-----------+------+
|Churn_Label| count|
+-----------+------+
|       NULL| 76563|
|          0|159430|
|          1| 68767|
+-----------+------+



Rows with NULL Churn_Label in train_df: 76563


In [33]:
# Cell 8 — Pick best model and save it with versioning (Priority 1)

# Choose best by AUC ROC
best_name, best_metrics, best_model, best_pred = max(results, key=lambda t: t[1]["auc_roc"])
print(f"🏆 Best model by AUC ROC: {best_name}")
print(f"   Metrics: {best_metrics}")

# Priority 1: Use sequential model versioning (v1.0, v2.0, v3.0, etc.)
STORAGE_ACCOUNT = storage_account1
CONTAINER_GOLD = "golddata"
MODELS_BASE_PATH = (
    f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"models/{USE_CASE}"
)

# Auto-detect next version using Spark Hadoop FileSystem API
try:
    # Get Hadoop FileSystem
    hadoop_fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jvm.org.apache.hadoop.fs.Path(MODELS_BASE_PATH).toUri(),
        spark._jsc.hadoopConfiguration()
    )
    
    # List existing version folders
    path_obj = spark._jvm.org.apache.hadoop.fs.Path(MODELS_BASE_PATH)
    status_array = hadoop_fs.listStatus(path_obj)
    
    existing_versions = []
    for status in status_array:
        folder_name = status.getPath().getName()
        if folder_name.startswith('v') and folder_name[1:].replace('.', '').isdigit():
            try:
                version_num = float(folder_name[1:])
                existing_versions.append(version_num)
            except:
                pass
    
    next_version = max(existing_versions) + 1.0 if existing_versions else 1.0
    version_str = f"v{next_version:.1f}"
    print(f"✅ Auto-detected existing versions: {sorted(existing_versions)}")
    print(f"   Next version: {version_str}")
except Exception as e:
    print(f"⚠️ Could not detect existing versions: {e}")
    version_str = "v1.0"

MODEL_PATH = f"{MODELS_BASE_PATH}/{version_str}"

print(f"\nSaving best model with version: {version_str}")
print(f"Path: {MODEL_PATH}")

best_model.write().overwrite().save(MODEL_PATH)

print("✅ Model saved with version tracking")

# Log final model metadata to MLflow (log metrics instead, params are immutable)
mlflow.log_metrics({
    "final_auc_roc": round(best_metrics["auc_roc"], 4),
    "final_auc_pr": round(best_metrics["auc_pr"], 4),
    "final_f1": round(best_metrics["f1"], 4),
    "final_accuracy": round(best_metrics["accuracy"], 4),
})

print(f"✅ Final model metrics logged to MLflow")

🏆 Best model by AUC ROC: RandomForest
   Metrics: {'auc_roc': 1.0, 'auc_pr': 1.0, 'f1': 1.0, 'accuracy': 1.0}
✅ Auto-detected existing versions: [1.0]
   Next version: v2.0

Saving best model with version: v2.0
Path: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/policy_churn/v2.0
✅ Auto-detected existing versions: [1.0]
   Next version: v2.0

Saving best model with version: v2.0
Path: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/policy_churn/v2.0


✅ Model saved with version tracking
✅ Final model metrics logged to MLflow


In [28]:
# Cell 9 — Load best model and score a few policies (serving simulation)

from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

# 1) Load the saved best model (path from Cell 8)
loaded_model = PipelineModel.load(MODEL_PATH)

# 2) Take a small random sample to mimic online scoring
sample_to_score = (
    test_df
        .orderBy(F.rand())
        .limit(20)
)

# 3) Apply same preprocessing + model
scored_raw = loaded_model.transform(prep_nulls(sample_to_score))

print("Scored schema:")
scored_raw.printSchema()

# 4) Convert probability vector -> array and pick P(churn=1)
scored = (
    scored_raw
        .withColumn("prob_arr", vector_to_array("probability"))
        .select(
            "Policy_ID",
            "Customer_ID",
            "Churn_Label",
            F.col("prob_arr")[1].alias("churn_prob"),   # probability of class 1
            "prediction",
            "Product_Line",
            "Channel",
            "Annual_Premium_GBP",
            "Tenure_Band",
            "Discount_Band"
        )
)

scored.show(truncate=False)


Scored schema:
root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = false)
 |-- Channel: string (nullable = false)
 |-- Sum_Insured_GBP: double (nullable = false)
 |-- Annual_Premium_GBP: double (nullable = false)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Policy_Duration_Days: double (nullable = false)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)
 |-- Tenure_Band: string (nullable = false)
 |-- Premium_Band: string (nullable = false)
 |-- Discount_Band: string (nullable = false)
 |-- Renewal_Outcome: string (nullable = false)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_discount_valid: integer (nullable = true)
 |-- dq_renewal_valid: integer (nullable = true)
 |-- Is_Renewal_Offered: integer (nullable = true)
 |-- Churn_Label: double (nul

In [29]:
# Cell 10 — OPTIONAL: log metrics into a Delta metrics table in gold

from datetime import datetime

METRICS_PATH = (
    f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"_metrics_ml_policy_churn"
)

now_ts = datetime.utcnow()

metrics_log_rows = [
    ("policy_churn", best_name, k, float(v), now_ts)
    for k, v in best_metrics.items()
]

metrics_log_df = spark.createDataFrame(
    metrics_log_rows,
    "model_family STRING, model_name STRING, metric STRING, value DOUBLE, ts TIMESTAMP"
)

(metrics_log_df
    .write
    .format("delta")
    .mode("append")
    .save(METRICS_PATH))

print("✅ Logged metrics to:", METRICS_PATH)

spark.read.format("delta").load(METRICS_PATH).orderBy(F.col("ts").desc()).show()


✅ Logged metrics to: abfss://golddata@clientdatastorage.dfs.core.windows.net/_metrics_ml_policy_churn


+------------+------------+--------+-----+--------------------+
|model_family|  model_name|  metric|value|                  ts|
+------------+------------+--------+-----+--------------------+
|policy_churn|RandomForest|accuracy|  1.0|2025-12-24 18:30:...|
|policy_churn|RandomForest| auc_roc|  1.0|2025-12-24 18:30:...|
|policy_churn|RandomForest|  auc_pr|  1.0|2025-12-24 18:30:...|
|policy_churn|RandomForest|      f1|  1.0|2025-12-24 18:30:...|
|policy_churn|RandomForest| auc_roc|  1.0|2025-12-23 18:40:...|
|policy_churn|RandomForest|accuracy|  1.0|2025-12-23 18:40:...|
|policy_churn|RandomForest|  auc_pr|  1.0|2025-12-23 18:40:...|
|policy_churn|RandomForest|      f1|  1.0|2025-12-23 18:40:...|
|policy_churn|RandomForest| auc_roc|  1.0|2025-12-23 18:25:...|
|policy_churn|RandomForest|  auc_pr|  1.0|2025-12-23 18:25:...|
|policy_churn|RandomForest|      f1|  1.0|2025-12-23 18:25:...|
|policy_churn|RandomForest|accuracy|  1.0|2025-12-23 18:25:...|
|policy_churn|RandomForest| auc_roc|  1.

In [34]:
# Cell 11 — Log full MLflow run with params, metrics, and model artifact

import mlflow
import mlflow.spark
from datetime import datetime

# We already did this in Cell 0b, but it's safe to repeat:
# (if you move this cell to another notebook later)
from pathlib import Path

PROJECT_ROOT = Path("/Users/manojrammopati/Public/Projects/bupa_insurance_project")
TRACKING_URI = f"file:{PROJECT_ROOT / 'mlruns'}"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("bupa_policy_churn")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

# Sanity check: we expect these to already exist from earlier cells
print("Best model:", best_name, "metrics:", best_metrics)

# End any active MLflow runs before starting a new one
mlflow.end_run()

with mlflow.start_run(run_name=f"policy_churn_{best_name.lower()}"):
    # ---- Params: data + features ----
    mlflow.log_param("use_case", "policy_churn")
    mlflow.log_param("label_col", label_col)                       # "Churn_Label"
    mlflow.log_param("numeric_features", ",".join(numeric_cols))
    mlflow.log_param("categorical_features", ",".join(categorical_cols))
    mlflow.log_param("train_rows", train_pre.count())
    mlflow.log_param("test_rows",  test_pre.count())
    mlflow.log_param("chosen_model", best_name)

    # ---- Params: model hyperparams (for the chosen model) ----
    if best_name == "LogisticRegression":
        mlflow.log_param("algo", "LogisticRegression")
        mlflow.log_param("maxIter",       lr.getOrDefault("maxIter"))
        mlflow.log_param("regParam",      lr.getOrDefault("regParam"))
        mlflow.log_param("elasticNet",    lr.getOrDefault("elasticNetParam"))
    elif best_name == "RandomForest":
        mlflow.log_param("algo", "RandomForest")
        mlflow.log_param("numTrees",      rf.getOrDefault("numTrees"))
        mlflow.log_param("maxDepth",      rf.getOrDefault("maxDepth"))
        mlflow.log_param("subsamplingRate", rf.getOrDefault("subsamplingRate"))
    elif best_name == "GBTClassifier":
        mlflow.log_param("algo", "GBTClassifier")
        mlflow.log_param("maxIter",       gbt.getOrDefault("maxIter"))
        mlflow.log_param("maxDepth",      gbt.getOrDefault("maxDepth"))
        mlflow.log_param("stepSize",      gbt.getOrDefault("stepSize"))

    # ---- Metrics: log metrics for all 3 models (prefixed) ----
    # `results` looks like: [("LogisticRegression", lr_metrics), ("RandomForest", rf_metrics), ...]
    for model_name, m, _, _ in results:
        prefix = f"{model_name}."
        mlflow.log_metric(prefix + "auc_roc",   float(m["auc_roc"]))
        mlflow.log_metric(prefix + "auc_pr",    float(m["auc_pr"]))
        mlflow.log_metric(prefix + "f1",        float(m["f1"]))
        mlflow.log_metric(prefix + "accuracy",  float(m["accuracy"]))

    # ---- Metrics: best model metrics as top-level ----
    mlflow.log_metric("best_auc_roc",   float(best_metrics["auc_roc"]))
    mlflow.log_metric("best_auc_pr",    float(best_metrics["auc_pr"]))
    mlflow.log_metric("best_f1",        float(best_metrics["f1"]))
    mlflow.log_metric("best_accuracy",  float(best_metrics["accuracy"]))

    # ---- Artifact: log the best Spark model into MLflow ----
    # (This is separate from the ADLS save you already do to MODEL_PATH.)
    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="model"   # will appear under this run in mlruns/
    )


print("✅ Logged MLflow run to experiment 'bupa_policy_churn'")
print("👉 You can open MLflow UI with:  mlflow ui --backend-store-uri", TRACKING_URI)

MLflow tracking URI: file:/Users/manojrammopati/Public/Projects/bupa_insurance_project/mlruns
Best model: RandomForest metrics: {'auc_roc': 1.0, 'auc_pr': 1.0, 'f1': 1.0, 'accuracy': 1.0}


✅ Logged MLflow run to experiment 'bupa_policy_churn'
👉 You can open MLflow UI with:  mlflow ui --backend-store-uri file:/Users/manojrammopati/Public/Projects/bupa_insurance_project/mlruns


In [16]:
from datetime import datetime
import utils_silver  # or `from utils_silver import write_ml_monitoring`

now_ts = datetime.utcnow()

rows = []

# Extract metrics from results list
# results = [(model_name, metrics, model, pred), ...]
model_names = ["LogisticRegression", "RandomForestClassifier", "GBTClassifier"]

for model_name, metrics_dict, _, _ in results:
    rows.append({
        "model_name":   model_name,
        "use_case":     "policy_churn",
        "dataset_name": "ft_policy_churn",
        "dataset_split": "test",
        "auc":       float(metrics_dict["auc_roc"]),
        "accuracy":  float(metrics_dict["accuracy"]),
        "precision": float(metrics_dict.get("precision", 0.0)),  # not logged, so default
        "recall":    float(metrics_dict.get("recall", 0.0)),     # not logged, so default
        "f1":        float(metrics_dict["f1"]),
        "ts":        now_ts,
        "notes":     f"{model_name} from cell 7 training",
    })


utils_silver.write_ml_monitoring(
    spark=spark,
    rows=rows,
    paths_gold=paths_gold,
)

print("✅ Logged all model metrics to ML monitoring")

[ML_MON] Wrote 3 rows to abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring
✅ Logged all model metrics to ML monitoring


In [17]:
df = spark.read.format("delta").load(
    "abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring"
)

df.printSchema()
df.show()


root
 |-- run_ts: timestamp (nullable = true)
 |-- model_name: string (nullable = true)
 |-- use_case: string (nullable = true)
 |-- dataset_name: string (nullable = true)
 |-- dataset_split: string (nullable = true)
 |-- auc: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- precision: double (nullable = true)
 |-- recall: double (nullable = true)
 |-- f1: double (nullable = true)
 |-- notes: string (nullable = true)




[Stage 1222:==============>                                         (1 + 3) / 4]



+--------------------+--------------------+----------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------+
|              run_ts|          model_name|        use_case|   dataset_name|dataset_split|               auc|          accuracy|         precision|             recall|                f1|               notes|
+--------------------+--------------------+----------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------+
|2025-12-10 21:02:...|        RandomForest|    claims_fraud| ft_claims_risk|         test| 0.987404471063458|0.7047248659786458|               1.0|0.18930766781769648|0.3183493606243436|Best of RandomFor...|
|2025-12-23 18:32:...|  LogisticRegression|    policy_churn|ft_policy_churn|         test|0.9999994178689208|               1.0|               0.0|                0.0| 